In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(project_root)

c:\dev\util


In [4]:
from src.data_fetcher import build_live_carbon_forecast_table

df = build_live_carbon_forecast_table(region="CAISO")
print(df.head())
print(df.columns)

REQUEST STATUS: 400
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"error":"INVALID_REGION","message":"You must provide a valid region. You may query /region-from-loc to determine the region for a given location","docs":"https://docs.watttime.org/#tag/GET-Regions-and-Maps/operation/get_reg_loc_v3_region_from_loc_get"}


HTTPError: 400 Client Error: Bad Request for url: https://api.watttime.org/v3/forecast?region=CAISO&signal_type=co2_moer

In [ ]:
from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

In [ ]:
from src.pipeline import run_util_pipeline
import inspect

print(run_util_pipeline)
print(inspect.signature(run_util_pipeline))
print(run_util_pipeline.__code__.co_filename)

<function run_util_pipeline at 0x0000022D66AF28E0>
(workload_input: 'Any', mapping_path: 'str | Path', carbon_path: 'str | Path | None' = None, price_path: 'str | Path | None' = None, forecast_mode: 'str' = 'demo', schedule_mode: 'str' = 'flexible', carbon_estimation_mode: 'str' = 'forecast_only', historical_days: 'int' = 7, current_time_override: 'str | None' = None) -> 'dict[str, Any]'
C:\dev\util\src\pipeline.py


In [ ]:
import pandas as pd
from pathlib import Path

from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

In [ ]:
def inspect_deadline_window(forecast_df, compute_hours_required, deadline):
    df = forecast_df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # infer interval
    diffs = df["timestamp"].diff().dropna()
    interval_minutes = diffs.dt.total_seconds().median() / 60
    rows_per_hour = 60 / interval_minutes
    slots_required = int(round(compute_hours_required * rows_per_hour))

    now_ts = pd.Timestamp.now()

    if df["timestamp"].max() < now_ts:
        effective_now_ts = df["timestamp"].min()
    else:
        effective_now_ts = now_ts

    deadline_ts = pd.to_datetime(deadline)
    if getattr(deadline_ts, "tzinfo", None) is not None:
        deadline_ts = deadline_ts.tz_localize(None)

    df["eligible_flag_debug"] = (
        (df["timestamp"] >= effective_now_ts) & (df["timestamp"] <= deadline_ts)
    ).astype(int)

    eligible_df = df[df["eligible_flag_debug"] == 1].copy()

    print("=" * 70)
    print("FORECAST WINDOW DEBUG")
    print("=" * 70)
    print(f"forecast min timestamp:   {df['timestamp'].min()}")
    print(f"forecast max timestamp:   {df['timestamp'].max()}")
    print(f"machine now timestamp:    {now_ts}")
    print(f"effective now timestamp:  {effective_now_ts}")
    print(f"deadline timestamp:       {deadline_ts}")
    print(f"interval minutes:         {interval_minutes}")
    print(f"rows per hour:            {rows_per_hour}")
    print(f"compute hours required:   {compute_hours_required}")
    print(f"slots required:           {slots_required}")
    print(f"eligible row count:       {len(eligible_df)}")

    if len(eligible_df) > 0:
        print(f"earliest eligible row:    {eligible_df['timestamp'].min()}")
        print(f"latest eligible row:      {eligible_df['timestamp'].max()}")
    else:
        print("earliest eligible row:    NONE")
        print("latest eligible row:      NONE")

    return df, eligible_df, {
        "interval_minutes": interval_minutes,
        "rows_per_hour": rows_per_hour,
        "slots_required": slots_required,
        "now_ts": now_ts,
        "effective_now_ts": effective_now_ts,
        "deadline_ts": deadline_ts,
    }

In [ ]:
def inspect_selected_rows(optimized_df, effective_now_ts, deadline_ts):
    df = optimized_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    selected = df[df["run_flag"] == 1].copy()

    print("=" * 70)
    print("SELECTION DEBUG")
    print("=" * 70)
    print(f"selected row count:       {len(selected)}")

    if len(selected) > 0:
        print(f"earliest selected row:    {selected['timestamp'].min()}")
        print(f"latest selected row:      {selected['timestamp'].max()}")
    else:
        print("earliest selected row:    NONE")
        print("latest selected row:      NONE")

    selected_after_deadline = selected[selected["timestamp"] > deadline_ts]
    selected_before_now = selected[selected["timestamp"] < effective_now_ts]

    print(f"selected rows after deadline:      {len(selected_after_deadline)}")
    print(f"selected rows before effective now:{len(selected_before_now)}")

    return selected, selected_after_deadline, selected_before_now

In [ ]:
workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-14 18:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
)

forecast_df = result["forecast"]
optimized_df = result["optimized"]

debug_df, eligible_df, debug_info = inspect_deadline_window(
    forecast_df=forecast_df,
    compute_hours_required=workload.compute_hours_required,
    deadline=workload.deadline,
)

selected_df, after_deadline_df, before_now_df = inspect_selected_rows(
    optimized_df=optimized_df,
    effective_now_ts=debug_info["effective_now_ts"],
    deadline_ts=debug_info["deadline_ts"],
)

selected_df[["timestamp", "carbon_g_per_kwh", "price_per_kwh", "score", "run_flag"]].head(20)

FORECAST WINDOW DEBUG
forecast min timestamp:   2026-03-13 00:00:00
forecast max timestamp:   2026-03-13 23:00:00
machine now timestamp:    2026-03-15 21:31:31.692449
effective now timestamp:  2026-03-13 00:00:00
deadline timestamp:       2026-03-14 18:00:00
interval minutes:         60.0
rows per hour:            1.0
compute hours required:   4
slots required:           4
eligible row count:       24
earliest eligible row:    2026-03-13 00:00:00
latest eligible row:      2026-03-13 23:00:00
SELECTION DEBUG
selected row count:       4
earliest selected row:    2026-03-13 13:00:00
latest selected row:      2026-03-13 16:00:00
selected rows after deadline:      0
selected rows before effective now:0


,timestamp,carbon_g_per_kwh,price_per_kwh,score,run_flag
13,2026-03-13 13:00:00,150,0.27,150,1
14,2026-03-13 14:00:00,140,0.30,140,1
15,2026-03-13 15:00:00,130,0.32,130,1
16,2026-03-13 16:00:00,140,0.34,140,1


In [ ]:
workload_block = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-14 18:00",
    objective="carbon",
    machine_watts=300,
)

result_block = run_util_pipeline(
    workload_input=workload_block,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="block",
    carbon_estimation_mode="forecast_only",
)

forecast_block_df = result_block["forecast"]
optimized_block_df = result_block["optimized"]

debug_df_block, eligible_df_block, debug_info_block = inspect_deadline_window(
    forecast_df=forecast_block_df,
    compute_hours_required=workload_block.compute_hours_required,
    deadline=workload_block.deadline,
)

selected_block_df, after_deadline_block_df, before_now_block_df = inspect_selected_rows(
    optimized_df=optimized_block_df,
    effective_now_ts=debug_info_block["effective_now_ts"],
    deadline_ts=debug_info_block["deadline_ts"],
)

selected_block_df[["timestamp", "carbon_g_per_kwh", "price_per_kwh", "score", "run_flag"]].head(100)

FORECAST WINDOW DEBUG
forecast min timestamp:   2026-03-13 00:00:00
forecast max timestamp:   2026-03-13 23:00:00
machine now timestamp:    2026-03-15 21:31:31.820124
effective now timestamp:  2026-03-13 00:00:00
deadline timestamp:       2026-03-14 18:00:00
interval minutes:         60.0
rows per hour:            1.0
compute hours required:   4
slots required:           4
eligible row count:       24
earliest eligible row:    2026-03-13 00:00:00
latest eligible row:      2026-03-13 23:00:00
SELECTION DEBUG
selected row count:       4
earliest selected row:    2026-03-13 13:00:00
latest selected row:      2026-03-13 16:00:00
selected rows after deadline:      0
selected rows before effective now:0


,timestamp,carbon_g_per_kwh,price_per_kwh,score,run_flag
13,2026-03-13 13:00:00,150,0.27,150,1
14,2026-03-13 14:00:00,140,0.30,140,1
15,2026-03-13 15:00:00,130,0.32,130,1
16,2026-03-13 16:00:00,140,0.34,140,1


In [ ]:
def check_block_contiguity(selected_df):
    selected = selected_df.copy().sort_values("timestamp").reset_index(drop=True)

    if len(selected) < 2:
        print("Not enough selected rows to test contiguity.")
        return

    diffs = selected["timestamp"].diff().dropna()
    diff_counts = diffs.value_counts().sort_index()

    print("=" * 70)
    print("BLOCK CONTIGUITY CHECK")
    print("=" * 70)
    print(diff_counts)

check_block_contiguity(selected_block_df)

BLOCK CONTIGUITY CHECK
timestamp
0 days 01:00:00    3
Name: count, dtype: int64


In [ ]:
workload_live = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-15 18:00",
    objective="carbon",
    machine_watts=300,
)

result_live = run_util_pipeline(
    workload_input=workload_live,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
)

forecast_live_df = result_live["forecast"]
optimized_live_df = result_live["optimized"]

debug_live_df, eligible_live_df, debug_info_live = inspect_deadline_window(
    forecast_df=forecast_live_df,
    compute_hours_required=workload_live.compute_hours_required,
    deadline=workload_live.deadline,
)

selected_live_df, after_deadline_live_df, before_now_live_df = inspect_selected_rows(
    optimized_df=optimized_live_df,
    effective_now_ts=debug_info_live["effective_now_ts"],
    deadline_ts=debug_info_live["deadline_ts"],
)

forecast_live_df[["timestamp", "carbon_g_per_kwh"]].tail(10)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
REQUEST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-16T04:30:00+00:00","value":998.3},{"point_time":"2026-03-16T04:35:00+00:00","value":998.3},{"point_time":"2026-03-16T04:40:00+00:00","value":995.8},{"point_time":"2026-03-16T04:45:00+00:00","value":991.3},{"point_time":"2026-03-16T04:50:00+00:00","value":990.2},{"point_time":"2026-03-16T04:55:00+00:00","value":1002.6},{"point_time":"2026-03-16T05:00:00+00:00","value":1006.1},{"point_time":"2026-03-16T05:05:00+00:00","value":1004.3},{"point_time":"2026-03-16T05:10:


ValueError: compute_hours_required exceeds the amount of forecast time available between now and the deadline

In [ ]:
workload_live_ext = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-16 18:00",
    objective="carbon",
    machine_watts=300,
)

result_live_ext = run_util_pipeline(
    workload_input=workload_live_ext,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_plus_historical_expectation",
    historical_days=7,
)

forecast_live_ext_df = result_live_ext["forecast"]
optimized_live_ext_df = result_live_ext["optimized"]

debug_live_ext_df, eligible_live_ext_df, debug_info_live_ext = inspect_deadline_window(
    forecast_df=forecast_live_ext_df,
    compute_hours_required=workload_live_ext.compute_hours_required,
    deadline=workload_live_ext.deadline,
)

selected_live_ext_df, after_deadline_live_ext_df, before_now_live_ext_df = inspect_selected_rows(
    optimized_df=optimized_live_ext_df,
    effective_now_ts=debug_info_live_ext["effective_now_ts"],
    deadline_ts=debug_info_live_ext["deadline_ts"],
)

forecast_live_ext_df[["timestamp", "carbon_g_per_kwh", "carbon_source"]].tail(30)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
REQUEST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-15T04:15:00+00:00","value":990.1},{"point_time":"2026-03-15T04:20:00+00:00","value":990.1},{"point_time":"2026-03-15T04:25:00+00:00","value":989.9},{"point_time":"2026-03-15T04:30:00+00:00","value":985.2},{"point_time":"2026-03-15T04:35:00+00:00","value":980.2},{"point_time":"2026-03-15T04:40:00+00:00","value":982.0},{"point_time":"2026-03-15T04:45:00+00:00","value":980.0},{"point_time":"2026-03-15T04:50:00+00:00","value":977.7},{"point_time":"2026-03-15T04:55:00+
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/historical?region=CAISO_NORTH&signal_type=co2_moer&start=2026-03-08T04%3A17%3A35.532884%2B00%3A00&end=2026-03-15T04%3A17%3A35.532884%2B00%3A00
REQUEST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-08T04:20:00+00:00","val

C:\dev\util\src\forecasting\carbon_blender.py:99: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([live_df, extension_df], ignore_index=True)


,timestamp,carbon_g_per_kwh,carbon_source
424,2026-03-16 15:35:00,242.285714,historical_expectation
425,2026-03-16 15:40:00,211.285714,historical_expectation
426,2026-03-16 15:45:00,50.285714,historical_expectation
427,2026-03-16 15:50:00,48.857143,historical_expectation
428,2026-03-16 15:55:00,50.000000,historical_expectation
429,2026-03-16 16:00:00,66.142857,historical_expectation
430,2026-03-16 16:05:00,65.857143,historical_expectation
431,2026-03-16 16:10:00,66.857143,historical_expectation
432,2026-03-16 16:15:00,61.000000,historical_expectation
433,2026-03-16 16:20:00,63.000000,historical_expectation


In [ ]:
# --- DEADLINE CLIPPING TEST ---

workload_deadline_test = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-13 10:00",   # deadline INSIDE the demo forecast window
    objective="carbon",
    machine_watts=300,
)

print("RUNNING FLEXIBLE MODE TEST")
print("="*60)

result_flexible = run_util_pipeline(
    workload_input=workload_deadline_test,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
)

forecast_df = result_flexible["forecast"]
optimized_df = result_flexible["optimized"]

debug_df, eligible_df, debug_info = inspect_deadline_window(
    forecast_df=forecast_df,
    compute_hours_required=workload_deadline_test.compute_hours_required,
    deadline=workload_deadline_test.deadline,
)

selected_df, after_deadline_df, before_now_df = inspect_selected_rows(
    optimized_df=optimized_df,
    effective_now_ts=debug_info["effective_now_ts"],
    deadline_ts=debug_info["deadline_ts"],
)

print("\nSelected rows:")
display(selected_df[["timestamp","score","run_flag"]])


print("\n\nRUNNING BLOCK MODE TEST")
print("="*60)

result_block = run_util_pipeline(
    workload_input=workload_deadline_test,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="block",
    carbon_estimation_mode="forecast_only",
)

optimized_block_df = result_block["optimized"]

selected_block_df, after_deadline_block_df, before_now_block_df = inspect_selected_rows(
    optimized_df=optimized_block_df,
    effective_now_ts=debug_info["effective_now_ts"],
    deadline_ts=debug_info["deadline_ts"],
)

print("\nBlock selected rows:")
display(selected_block_df[["timestamp","score","run_flag"]])

RUNNING FLEXIBLE MODE TEST
FORECAST WINDOW DEBUG
forecast min timestamp:   2026-03-13 00:00:00
forecast max timestamp:   2026-03-13 23:00:00
machine now timestamp:    2026-03-14 21:27:05.973224
effective now timestamp:  2026-03-13 00:00:00
deadline timestamp:       2026-03-13 10:00:00
interval minutes:         60.0
rows per hour:            1.0
compute hours required:   4
slots required:           4
eligible row count:       11
earliest eligible row:    2026-03-13 00:00:00
latest eligible row:      2026-03-13 10:00:00
SELECTION DEBUG
selected row count:       4
earliest selected row:    2026-03-13 07:00:00
latest selected row:      2026-03-13 10:00:00
selected rows after deadline:      0
selected rows before effective now:0

Selected rows:


,timestamp,score,run_flag
7,2026-03-13 07:00:00,270.0,1
8,2026-03-13 08:00:00,220.0,1
9,2026-03-13 09:00:00,210.0,1
10,2026-03-13 10:00:00,200.0,1




RUNNING BLOCK MODE TEST
SELECTION DEBUG
selected row count:       4
earliest selected row:    2026-03-13 07:00:00
latest selected row:      2026-03-13 10:00:00
selected rows after deadline:      0
selected rows before effective now:0

Block selected rows:


,timestamp,score,run_flag
7,2026-03-13 07:00:00,270.0,1
8,2026-03-13 08:00:00,220.0,1
9,2026-03-13 09:00:00,210.0,1
10,2026-03-13 10:00:00,200.0,1


In [ ]:
workload_now_test = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-13 10:00",
    objective="carbon",
    machine_watts=300,
)

result_now_test = run_util_pipeline(
    workload_input=workload_now_test,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
    current_time_override="2026-03-13 06:00",
)

forecast_df = result_now_test["forecast"]
optimized_df = result_now_test["optimized"]

debug_df, eligible_df, debug_info = inspect_deadline_window(
    forecast_df=forecast_df,
    compute_hours_required=workload_now_test.compute_hours_required,
    deadline=workload_now_test.deadline,
)

# overwrite debug_info manually to match forced now
debug_info["effective_now_ts"] = pd.Timestamp("2026-03-13 06:00")
debug_info["deadline_ts"] = pd.Timestamp("2026-03-13 10:00")

selected_df, after_deadline_df, before_now_df = inspect_selected_rows(
    optimized_df=optimized_df,
    effective_now_ts=debug_info["effective_now_ts"],
    deadline_ts=debug_info["deadline_ts"],
)

display(selected_df[["timestamp", "score", "run_flag"]])

TypeError: run_util_pipeline() got an unexpected keyword argument 'current_time_override'

In [ ]:
import importlib
import src.pipeline
import src.optimizer

importlib.reload(src.optimizer)
importlib.reload(src.pipeline)

from src.pipeline import run_util_pipeline

print(run_util_pipeline)

<function run_util_pipeline at 0x0000022D43EE6520>


In [ ]:
import inspect
print(inspect.signature(run_util_pipeline))

(workload_input: 'Any', mapping_path: 'str | Path', carbon_path: 'str | Path | None' = None, price_path: 'str | Path | None' = None, forecast_mode: 'str' = 'demo', schedule_mode: 'str' = 'flexible', carbon_estimation_mode: 'str' = 'forecast_only', historical_days: 'int' = 7, current_time_override: 'str | None' = None) -> 'dict[str, Any]'


In [ ]:
workload_now_test = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-13 10:00",
    objective="carbon",
    machine_watts=300,
)

result_now_test = run_util_pipeline(
    workload_input=workload_now_test,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
    current_time_override="2026-03-13 06:00",
)

forecast_df = result_now_test["forecast"]
optimized_df = result_now_test["optimized"]

selected_df = optimized_df[optimized_df["run_flag"] == 1].copy()
selected_df["timestamp"] = pd.to_datetime(selected_df["timestamp"])

print(selected_df[["timestamp", "score", "run_flag"]])

             timestamp  score  run_flag
7  2026-03-13 07:00:00  270.0         1
8  2026-03-13 08:00:00  220.0         1
9  2026-03-13 09:00:00  210.0         1
10 2026-03-13 10:00:00  200.0         1


In [ ]:
workload_now_test_block = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-13 10:00",
    objective="carbon",
    machine_watts=300,
)

result_now_test_block = run_util_pipeline(
    workload_input=workload_now_test_block,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="block",
    carbon_estimation_mode="forecast_only",
    current_time_override="2026-03-13 06:00",
)

optimized_block_df = result_now_test_block["optimized"].copy()
optimized_block_df["timestamp"] = pd.to_datetime(optimized_block_df["timestamp"])

selected_block_df = optimized_block_df[optimized_block_df["run_flag"] == 1].copy()

print(selected_block_df[["timestamp", "score", "run_flag"]])

             timestamp  score  run_flag
7  2026-03-13 07:00:00  270.0         1
8  2026-03-13 08:00:00  220.0         1
9  2026-03-13 09:00:00  210.0         1
10 2026-03-13 10:00:00  200.0         1


In [ ]:
opt_df = result_now_test_block["optimized"].copy()
opt_df["timestamp"] = pd.to_datetime(opt_df["timestamp"])

eligible_window = opt_df[
    (opt_df["timestamp"] >= pd.Timestamp("2026-03-13 06:00")) &
    (opt_df["timestamp"] <= pd.Timestamp("2026-03-13 10:00"))
].copy()

print("Eligible window:")
print(eligible_window[["timestamp", "score", "run_flag"]])

print("\nSelected block:")
selected_block = eligible_window[eligible_window["run_flag"] == 1].copy()
print(selected_block[["timestamp", "score", "run_flag"]])

print("\nTotal selected block score:", selected_block["score"].sum())

Eligible window:
             timestamp  score  run_flag
6  2026-03-13 06:00:00  280.0         0
7  2026-03-13 07:00:00  270.0         1
8  2026-03-13 08:00:00  220.0         1
9  2026-03-13 09:00:00  210.0         1
10 2026-03-13 10:00:00  200.0         1

Selected block:
             timestamp  score  run_flag
7  2026-03-13 07:00:00  270.0         1
8  2026-03-13 08:00:00  220.0         1
9  2026-03-13 09:00:00  210.0         1
10 2026-03-13 10:00:00  200.0         1

Total selected block score: 900.0


In [ ]:
window_df = result_now_test_block["optimized"].copy()
window_df["timestamp"] = pd.to_datetime(window_df["timestamp"])

window_df = window_df[
    (window_df["timestamp"] >= pd.Timestamp("2026-03-13 06:00")) &
    (window_df["timestamp"] <= pd.Timestamp("2026-03-13 10:00"))
].copy().sort_values("timestamp").reset_index(drop=True)

slots_required = 4

candidates = []

for i in range(len(window_df) - slots_required + 1):
    block = window_df.iloc[i:i+slots_required].copy()
    candidates.append({
        "start": block["timestamp"].iloc[0],
        "end": block["timestamp"].iloc[-1],
        "total_score": block["score"].sum()
    })

candidates_df = pd.DataFrame(candidates)
print(candidates_df.sort_values("total_score"))

                start                 end  total_score
1 2026-03-13 07:00:00 2026-03-13 10:00:00        900.0
0 2026-03-13 06:00:00 2026-03-13 09:00:00        980.0


In [ ]:
import importlib
import src.optimizer
import src.pipeline

importlib.reload(src.optimizer)
importlib.reload(src.pipeline)

from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput
import pandas as pd

workload_flex_test = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-13 10:00",
    objective="carbon",
    machine_watts=300,
)

result_flex_test = run_util_pipeline(
    workload_input=workload_flex_test,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
    current_time_override="2026-03-13 06:00",
)

opt_flex_df = result_flex_test["optimized"].copy()
opt_flex_df["timestamp"] = pd.to_datetime(opt_flex_df["timestamp"])

eligible_window = opt_flex_df[
    (opt_flex_df["timestamp"] >= pd.Timestamp("2026-03-13 06:00")) &
    (opt_flex_df["timestamp"] <= pd.Timestamp("2026-03-13 10:00"))
].copy()

selected_flex = opt_flex_df[opt_flex_df["run_flag"] == 1].copy()

print("Eligible window:")
print(eligible_window[["timestamp", "carbon_g_per_kwh", "score", "eligible_flag", "run_flag"]])

print("\nSelected flexible rows:")
print(selected_flex[["timestamp", "carbon_g_per_kwh", "score", "run_flag"]])

print("\nSelected row count:", len(selected_flex))
print("Any selected before 06:00:", (selected_flex["timestamp"] < pd.Timestamp("2026-03-13 06:00")).any())
print("Any selected after 10:00:", (selected_flex["timestamp"] > pd.Timestamp("2026-03-13 10:00")).any())

Eligible window:
             timestamp  carbon_g_per_kwh  score  eligible_flag  run_flag
6  2026-03-13 06:00:00               280  280.0              1         0
7  2026-03-13 07:00:00               270  270.0              1         1
8  2026-03-13 08:00:00               220  220.0              1         1
9  2026-03-13 09:00:00               210  210.0              1         1
10 2026-03-13 10:00:00               200  200.0              1         1

Selected flexible rows:
             timestamp  carbon_g_per_kwh  score  run_flag
7  2026-03-13 07:00:00               270  270.0         1
8  2026-03-13 08:00:00               220  220.0         1
9  2026-03-13 09:00:00               210  210.0         1
10 2026-03-13 10:00:00               200  200.0         1

Selected row count: 4
Any selected before 06:00: False
Any selected after 10:00: False


In [ ]:
import importlib
import src.optimizer
import src.pipeline

importlib.reload(src.optimizer)
importlib.reload(src.pipeline)

<module 'src.pipeline' from 'C:\\dev\\util\\src\\pipeline.py'>

In [ ]:
import datetime

forecast = result["forecast"].copy()
forecast["timestamp"] = pd.to_datetime(forecast["timestamp"])

print("Current time:", datetime.datetime.now())
print("Forecast start:", forecast["timestamp"].min())
print("Forecast end:", forecast["timestamp"].max())

ModuleNotFoundError: No module named 'pgeocode'

In [ ]:
pip install pgeocode

  Using cached pgeocode-0.5.0-py3-none-any.whl.metadata (7.9 kB)
Using cached pgeocode-0.5.0-py3-none-any.whl (9.8 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:

from src.location.zip_resolver import zip_to_coordinates

result = zip_to_coordinates("93106")
print(result)

{'zip_code': '93106', 'latitude': 34.4329, 'longitude': -119.8371}


In [ ]:
from services.watttime_service import get_ba_from_loc

result = get_ba_from_loc(34.4329, -119.8371)
print(result)

ModuleNotFoundError: No module named 'services'

In [ ]:
from pathlib import Path

path = Path(r"C:\dev\util\services\watttime_service.py")
text = path.read_text(encoding="utf-8")

print("get_ba_from_loc" in text)
print(path)

In [ ]:
from pathlib import Path

path = Path(r"C:\dev\util\services\watttime_service.py")
text = path.read_text(encoding="utf-8")

print("get_ba_from_loc" in text)
print(path)

True
C:\dev\util\services\watttime_service.py


In [ ]:
import os
os.chdir(r"C:\dev\util")
print(os.getcwd())

C:\dev\util


In [ ]:
import services.watttime_service as ws

print(ws.__file__)
print(hasattr(ws, "get_ba_from_loc"))

C:\dev\util\services\watttime_service.py
True


In [ ]:
import sys
project_root = r"C:\dev\util"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[:3])

['C:\\dev\\util', 'c:\\Users\\Finn Case\\miniconda3\\envs\\util\\python312.zip', 'c:\\Users\\Finn Case\\miniconda3\\envs\\util\\DLLs']


In [ ]:
import os
import sys

os.chdir(r"C:\dev\util")
if r"C:\dev\util" not in sys.path:
    sys.path.insert(0, r"C:\dev\util")

import services.watttime_service as ws

print(ws.__file__)
print(hasattr(ws, "get_ba_from_loc"))

C:\dev\util\services\watttime_service.py
True


In [ ]:
from services.watttime_service import get_ba_from_loc
print(get_ba_from_loc(34.4329, -119.8371))

REQUEST STATUS: 200
REQUEST CONTENT TYPE: text/html
REQUEST URL: https://docs.watttime.org?latitude=34.4329&longitude=-119.8371
REQUEST RESPONSE PREVIEW: <!DOCTYPE html>
<html>
  <head>
    <title>WattTime Data API</title>
    <link rel="icon" type="image/x-icon" href="/favicon.ico">
    <!-- needed for adaptive design -->
    <meta charset="utf-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link href="https://fonts.googleapis.com/css?family=Montserrat:300,400,700|Roboto:300,400,700" rel="stylesheet">

    <!--
    ReDoc doesn't change outer page styles
    -->
    <style>
      body {
        margin: 0;
      


ValueError: WattTime endpoint did not return JSON.

In [ ]:
import os
import sys
import requests
from requests.auth import HTTPBasicAuth

os.chdir(r"C:\dev\util")
if r"C:\dev\util" not in sys.path:
    sys.path.insert(0, r"C:\dev\util")

from services import watttime_service as ws

print("MODULE FILE:", ws.__file__)
print("BA URL:", ws.BA_FROM_LOC_URL)

token = ws.get_token()
print("TOKEN OK:", isinstance(token, str) and len(token) > 20)

response = requests.get(
    ws.BA_FROM_LOC_URL,
    headers={"Authorization": f"Bearer {token}"},
    params={"latitude": 34.4329, "longitude": -119.8371},
    timeout=60,
    allow_redirects=False,
)

print("STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("content-type"))
print("LOCATION HEADER:", response.headers.get("Location"))
print("FINAL URL:", response.url)
print("BODY PREVIEW:", response.text[:500])

MODULE FILE: C:\dev\util\services\watttime_service.py
BA URL: https://api2.watttime.org/v2/ba-from-loc
TOKEN OK: True
STATUS: 302
CONTENT TYPE: text/html
LOCATION HEADER: https://docs.watttime.org?latitude=34.4329&longitude=-119.8371
FINAL URL: https://api2.watttime.org/v2/ba-from-loc?latitude=34.4329&longitude=-119.8371
BODY PREVIEW: <html>
<head><title>302 Found</title></head>
<body>
<center><h1>302 Found</h1></center>
<hr><center>nginx</center>
</body>
</html>



In [ ]:
import sys
import os

PROJECT_ROOT = r"C:\dev\util"

os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root set to:", PROJECT_ROOT)

Project root set to: C:\dev\util


In [ ]:
from services.watttime_service import get_region_from_loc

result = get_region_from_loc(34.4329, -119.8371)
print(result)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
{'region': 'CAISO_SANBERNARDINO', 'region_full_name': 'California ISO San Bernardino', 'signal_type': 'co2_moer'}


In [ ]:
from src.location.location_service import resolve_zip_to_watttime_region

result = resolve_zip_to_watttime_region("93106")
print(result)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
{'zip_code': '93106', 'latitude': 34.4329, 'longitude': -119.8371, 'watttime_region': 'CAISO_SANBERNARDINO', 'watttime_region_full_name': 'California ISO San Bernardino', 'signal_type': 'co2_moer', 'raw_response': {'region': 'CAISO_SANBERNARDINO', 'region_full_name': 'California ISO San Bernardino', 'signal_type': 'co2_moer'}}


In [ ]:
from services.watttime_service import get_region_from_loc

print(get_region_from_loc(34.4329, -119.8371))
print(get_region_from_loc(34.4208, -119.6982))  # Santa Barbara-ish
print(get_region_from_loc(34.0522, -118.2437))  # Los Angeles

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
{'region': 'CAISO_SANBERNARDINO', 'region_full_name': 'California ISO San Bernardino', 'signal_type': 'co2_moer'}
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4208&longitude=-119.6982&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
{'region': 'CAISO_SANBERNARDINO', 'region_full_name': 'California ISO San Bernardino', 'signal_type': 'co2_moer'}
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST UR

In [ ]:
import os
import sys

PROJECT_ROOT = r"C:\dev\util"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=4,
    deadline="2026-03-16 12:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
    schedule_mode="flexible",
    carbon_estimation_mode="forecast_only",
)

print("Resolved region:", result["region"])
print("Location info:", result["location_info"])
print(result["forecast"].head())

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-16T04:55:00+00:00","value":1011.9},{"point_time":"2026-03-16T05:00:00+00:00","value":1011.9},{"point_time":"2026-03-16T05:05:00+00:00","value":1010.0},{"point_time":"2026-03-16T05:10:00+00:00","value":1008.3},{"point_time":"2026-03-16T05:15:00+00:00","value":1010.1},{"point_time":"2026-03-16T05:20:00+00:00","value":1006.1},{"point_time":"2026-03-16T05:25:00+00:00","value":1007.3},{"point_time":"2026-03-16T05:30:00+00:00","val

In [ ]:
result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

print("Resolved region:", result["region"])

NameError: name 'run_util_pipeline' is not defined

In [ ]:
import os
import sys
import inspect

PROJECT_ROOT = r"C:\dev\util"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.data_fetcher as dfm

print(dfm.__file__)
print(inspect.getsource(dfm.build_live_carbon_forecast_table))

C:\dev\util\src\data_fetcher.py
def build_live_carbon_forecast_table(
    region: str,
    placeholder_price_per_kwh: float = 0.15,
    carbon_estimation_mode: str = "forecast_only",
    historical_days: int = 7,
    deadline: str | None = None,
) -> pd.DataFrame:
    """
    Build a forecast table using live carbon forecast data from WattTime
    and a placeholder electricity price.

    carbon_estimation_mode:
    - forecast_only
    - forecast_plus_historical_expectation
    """

    # THIS IS THE FIX: use the region passed from the pipeline
    watttime_region = region

    carbon_df = get_watttime_forecast(watttime_region)

    required_columns = {"timestamp", "carbon_g_per_kwh"}
    if not required_columns.issubset(carbon_df.columns):
        raise ValueError(
            f"WattTime forecast must contain columns: {required_columns}"
        )

    carbon_df = _normalize_timestamp_column(carbon_df, "timestamp")

    if carbon_estimation_mode == "forecast_plus_historical_expectatio

In [ ]:
import inspect
import services.watttime_service as ws

print(ws.__file__)
print(inspect.getsource(ws.get_watttime_forecast))
print(inspect.getsource(ws.get_forecast))

C:\dev\util\services\watttime_service.py
def get_watttime_forecast(
    region: str = "CAISO_NORTH",
    signal_type: str = "co2_moer",
) -> pd.DataFrame:
    """
    Fetch live WattTime forecast data and return it in Util's standardized dataframe format.
    """
    forecast_json = get_forecast(region=region, signal_type=signal_type)
    return forecast_to_dataframe(forecast_json)

def get_forecast(
    region: str = "CAISO_NORTH",
    signal_type: str = "co2_moer",
) -> dict[str, Any]:
    """
    Fetch live carbon forecast JSON from WattTime.
    """
    params = {
        "region": region,
        "signal_type": signal_type,
    }
    return _fetch_json(FORECAST_URL, params)



In [ ]:
def get_watttime_forecast(region: str = "CAISO_NORTH", signal_type: str = "co2_moer"):
    forecast_json = get_forecast(region=region, signal_type=signal_type)
    return forecast_to_dataframe(forecast_json)

In [ ]:
from services.watttime_service import get_watttime_forecast

df = get_watttime_forecast("CAISO_SANBERNARDINO")
print(df.head())

REQUEST STATUS: 403
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_SANBERNARDINO&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"error":"INVALID_SCOPE","message":"You do not have sufficient access to this resource. See https://watttime.org/docs-dev/data-plans/ for more information or email contact@watttime.org for pricing. Please note that all users are allowed free access to the CAISO_NORTH region.","docs":"https://docs.watttime.org/"}


ValueError: WattTime request failed: forbidden (403).

In [5]:
from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

In [7]:
workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-16 18:00",
    objective="carbon",
    machine_watts=300,
)

In [9]:
result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
REQUEST STATUS: 403
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_SANBERNARDINO&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"error":"INVALID_SCOPE","message":"You do not have sufficient access to this resource. See https://watttime.org/docs-dev/data-plans/ for more information or email contact@watttime.org for pricing. Please note that all users are allowed free access to the CAISO_NORTH region.","docs":"https://docs.watttime.org/"}
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type

In [10]:
result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json; charset=utf-8
REQUEST URL: https://api.watttime.org/v3/region-from-loc?latitude=34.4329&longitude=-119.8371&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"region":"CAISO_SANBERNARDINO","region_full_name":"California ISO San Bernardino","signal_type":"co2_moer"}
REQUEST STATUS: 403
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_SANBERNARDINO&signal_type=co2_moer
REQUEST HISTORY: []
REQUEST RESPONSE PREVIEW: {"error":"INVALID_SCOPE","message":"You do not have sufficient access to this resource. See https://watttime.org/docs-dev/data-plans/ for more information or email contact@watttime.org for pricing. Please note that all users are allowed free access to the CAISO_NORTH region.","docs":"https://docs.watttime.org/"}
REQUEST STATUS: 200
REQUEST CONTENT TYPE: application/json
REQUEST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type

In [11]:
print(result["forecast"][[
    "forecast_region_requested",
    "forecast_region_used",
    "forecast_access_mode"
]].drop_duplicates())

  forecast_region_requested forecast_region_used forecast_access_mode
0       CAISO_SANBERNARDINO          CAISO_NORTH     preview_fallback
